## Connections and Pore Windows 


*Note: These tutorials are not a substitute for reading the paper. For details on the algorithms working under the hood, please read the original publication. However, these tutorials are useful for learning the syntax of mofography*

In this tutorial, we will learn how to find the connections between the different cavities in a material, locate the one or more adjoining pore windows, and to finally visualize them along with everything else that we have seen so far.

Finding the pore windows is an involved calculation, hence we need everything we have computed so far to get the pore windows, we need
- The distance grid
- the local maxima of the distance grid
- the region segmentation grid
- the distance values at the local maxima
- the pore graph

To give some brief insights into the window finding algorithm in mofography, we locate the windows as the centroid of the points in the shared outer boundary of two touching regions. There are couple of nuances to window finding,

- The windows can be fully internal to the unit cell or across periodic boundaries, mofography can detect and distinguich both of these.
- There can be multiple windows between two touching regions, mofography can detect these as long as these windows are spatially separated by at least 1 Å. Under the hood, mofography uses agglomerative clustering to group the boundary points before taking the centroid to account for this.


As before, we won't be explaining the steps that we have already covered in the previous tutorials, if anything is unclear, please refer to the previous tutorials. 



## Imports

In [ ]:
import sys, os 
path_to_mgr = os.path.abspath('c:/Users/Arun Gopalan/OneDrive - The University of Manchester/my_cloned_repos/mofography_general/mofography')
sys.path.append(path_to_mgr)
import mofography as mgr 


## Read in the structure 

In [ ]:
from ase.io import read 
path_to_cif=os.path.join(path_to_mgr,'examples', 'TestCIFs','P1-IRMOF-1.cif')
mat_atoms = read(path_to_cif)
mat_updated = mgr.update_with_covalent_bonds(mat_atoms)


## Compute the distance grid

In [ ]:
dgrid = mgr.dgrid_from_atoms(mat_atoms, spacing=0.5)

## Region segmenation

In [ ]:
regions, maxima = mgr.regions_from_dgrid(dgrid.compute())

## Get the radii of the maxima


In [ ]:
maxima_radii = dgrid.compute()[tuple(maxima.T)]

## Region adjacency graph

#### *Note: (Not recommended, but, faster)*
You can compute the RAG first from the regions grid, but, this can often lead to errors due to some algorithmic details within skimage, but, it is the faster option, and will work for a lot of materials. See below for a more robust option, where we compute the 
connections first and then use that to construct the RAG.


In [ ]:
rag  = mgr.get_rag_from_regions_and_grid(regions, dgrid.compute(), mat_atoms, maxima=maxima)

## Get the windows and connections

We have seen all the steps so far. If you have the pore graph already, we can locate the windows between all the connected regions using mofography 

In [ ]:
connections = mgr.get_connections_for_rag(rag, region_labels=regions, ase_atoms=mat_atoms, maxima=maxima, dgrid=dgrid.compute())

Now let's take a look at this connections variable.

In [ ]:
connections

The 'connections' variable is a list of lists. Each entry in has 4 elements.
- The labels of the two regions that are connected
- Confirmation flag
- Nature of the windows, whether it is internal or across periodic boundaries
- The coordinates of the window(s)
- The radius of the window in Å

### More robust option (Recommended)
In some materials, there is only one region, and hence no internal connections. In such cases, the 'get_rag_from_regions_and_dgrid' function that use the skimage backend will give and error. Although it is a bit slower, mofography includes to much more robust 
'connections_from_regions_and_dgrid' function to compute the windows and connections first, and then use it to construct the pore graph. Let's see how:

In [ ]:
connections = mgr.connections_from_regions_and_dgrid(regions, dgrid.compute(), maxima, mat_atoms)

which is the same as before: 

In [ ]:
connections

and then we can construct the pore graph like this:

In [ ]:
rag = mgr.rag_from_connections(connections, maxima)
#* let's add some properties to the RAG
shape = dgrid.shape
rag = mgr.add_maxima_to_rag(rag, maxima, maxima_radii, shape, mat_atoms)

## Visualize the windows

It is very useful to visualize the windows along with all the other information that we have already seen so far. Let's start by creating plot traces for the 
- framework
- the distance grid isosurface (colored by the regions)
- the local maxima
- the pore graph

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
cscale  = px.colors.qualitative.Pastel_r

frame_trace = mgr.plot_framework_as_mesh(mat_updated, stylize=True)
cell_trace = mgr.plot_cell(mat_updated)
isosurface_trace =  mgr.plot_surface_dgrid(dgrid.compute(), isovalue=1.75, ase_atoms  = mat_updated, opacity=0.7, region_labels=regions, cscale=cscale)
maxima_trace = mgr.plot_maxima(dgrid.compute(), maxima,mat_updated)
rag_trace = mgr.plot_rag(rag, name='pore graph')

mofography offers a wrapper function to plot the windwos from the connections variable

In [ ]:
windows_trace = mgr.plot_windows(connections, marker_color='red')

Now let's visualize everything in one figure. As before please change 'renderer' to 'notebook_connected' if you'd like to see the interactive plot in the notebook. Use 'renderer='browser'' to open the plot in a browser tab.

In [ ]:
fig = go.Figure(data=[*frame_trace, cell_trace, isosurface_trace, maxima_trace, rag_trace, windows_trace])
fig.update(layout=mgr.dark_layout)
fig.show(renderer='png')

Hover the mouse over the scatter points marking the windows to see all the information about the windows.

## Where to go from here?

Detecting windows is somewhat of a branch off of the main mofography workflow. Feel free to check out the Pockets detection tutorial if you haven't already. The tutorial on Paths detection is also another solid option. You may also want to check out the rest of the advanced tutorials.